# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data


A model that can estimate how much something costs, from its description.

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 4: Neural Networks and LLMs

Today we'll work from Traditional ML to Neural Networks to Large Language Models!!

In [1]:
# imports

import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.evaluator import evaluate
from litellm import completion
from pricer.items import Item
import numpy as np
from tqdm.notebook import tqdm
import csv
from sklearn.feature_extraction.text import HashingVectorizer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import CosineAnnealingLR


In [ ]:
LITE_MODE = False

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [5]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

README.md:   0%|          | 0.00/735 [00:00<?, ?B/s]

d:\Udemy\AI Engineer Core Track LLM Engineering, RAG, QLoRA, Agents\llm_engineering\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\djoshi\.cache\huggingface\hub\datasets--ed-donner--items_lite. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' 

train-00000-of-00001.parquet:   0%|          | 0.00/6.07M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


validation-00000-of-00001.parquet:   0%|          | 0.00/304k [00:00<?, ?B/s]

...

test-00000-of-00001.parquet:   0%|          | 0.00/304k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Loaded 20,000 training items, 1,000 validation items, 1,000 test items


# Before we look at the Artificial Neural Networks

## There is a different kind of Neural Network we could consider

In [6]:
# Write the test set to a CSV

# with open('human_in.csv', 'w', encoding="utf-8") as csvfile:
#     writer = csv.writer(csvfile)
#     for t in test[:100]:
#         writer.writerow([t.summary, 0])

with open('human_in.csv','w', encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    for t in test[:100]:
        writer.writerow([t.summary,0])

In [7]:
# Read it back in

human_predictions = []
with open('human_out.csv', 'r', encoding="utf-8") as csvfile:
    reader = csv.reader(csvfile)
    for row in reader:
        human_predictions.append(float(row[1]))

In [8]:
def human_pricer(item):
    idx = test.index(item)
    return human_predictions[idx]

In [9]:
human = human_pricer(test[0])
actual = test[0].price
print(f"Human predicted {human} for an item that actually costs {actual}")


Human predicted 120.0 for an item that actually costs 219.0


In [10]:
evaluate(human_pricer, test, size=100)

  0%|          | 0/100 [00:00<?, ?it/s]

$99 $184 $12 $15 $18 $10 $119 $135 $6 $270 $643 $329 $15 $26 $24 $18 $29 $25 $25 $53 $35 $126 $25 $127 $273 $398 $55 $6 $101 $51 $30 $5 $35 $9 $10 $419 $25 $11 $186 $33 $161 $51 $23 $155 $150 $4 $31 $18 $115 $82 $25 $111 $410 $75 $67 $34 $8 $10 $122 $28 $116 $17 $19 $60 $599 $60 $160 $355 $75 $34 $17 $2 $70 $76 $41 $9 $226 $5 $5 $4 $0 $7 $5 $74 $7 $10 $68 $74 $5 $3 $17 $45 $5 $16 $0 $153 $2 $122 $150 $355 

# And now - a vanilla Neural Network

During the remainder of this course we will get deeper into how Neural Networks work, and how to train a neural network.

This is just a sneak preview - let's make our own Neural Network, from scratch, using Pytorch.

Use this to get intuition; it's not important to know all about Neural networks at this point..

In [11]:
# Prepare our documents and prices

y = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [12]:
# Use the HashingVectorizer for a Bag of Words model
# Using binary=True with the CountVectorizer makes "one-hot vectors"

np.random.seed(42)
vectorizer = HashingVectorizer(n_features=5000, stop_words='english', binary=True)
X = vectorizer.fit_transform(documents)

In [13]:
# Define the neural network - here is Pytorch code to create a 8 layer neural network

class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 64)
        self.layer4 = nn.Linear(64, 64)
        self.layer5 = nn.Linear(64, 64)
        self.layer6 = nn.Linear(64, 64)
        self.layer7 = nn.Linear(64, 64)
        self.layer8 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        output1 = self.relu(self.layer1(x))
        output2 = self.relu(self.layer2(output1))
        output3 = self.relu(self.layer3(output2))
        output4 = self.relu(self.layer4(output3))
        output5 = self.relu(self.layer5(output4))
        output6 = self.relu(self.layer6(output5))
        output7 = self.relu(self.layer7(output6))
        output8 = self.layer8(output7)
        return output8

In [14]:
# Convert data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train_tensor, y_train_tensor, test_size=0.01, random_state=42)

# Create the loader
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Initialize the model
input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)

In [15]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Number of trainable parameters: {trainable_params:,}")

Number of trainable parameters: 669,249


In [16]:
# Define loss function and optimizer

loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# We will do 2 complete runs through the data

EPOCHS = 2

for epoch in range(EPOCHS):
    model.train()
    for batch_X, batch_y in tqdm(train_loader):
        optimizer.zero_grad()

        # The next 4 lines are the 4 stages of training: forward pass, loss calculation, backward pass, optimize
        
        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = loss_function(val_outputs, y_val)

    print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {loss.item():.3f}, Val Loss: {val_loss.item():.3f}')

  0%|          | 0/310 [00:00<?, ?it/s]

Epoch [1/2], Train Loss: 44528.793, Val Loss: 20128.818


  0%|          | 0/310 [00:00<?, ?it/s]

Epoch [2/2], Train Loss: 17385.920, Val Loss: 17630.232


In [17]:
def neural_network(item):
    model.eval()
    with torch.no_grad():
        vector = vectorizer.transform([item.summary])
        vector = torch.FloatTensor(vector.toarray())
        result = model(vector)[0].item()
    return max(0, result)

In [18]:
evaluate(neural_network, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$73 $52 $12 $20 $53 $176 $31 $56 $37 $37 $458 $128 $101 $158 $41 $37 $22 $19 $82 $15 $16 $35 $64 $49 $283 $273 $236 $30 $38 $37 $44 $137 $31 $27 $82 $263 $48 $127 $112 $66 $155 $111 $17 $95 $83 $59 $80 $73 $1 $17 $22 $41 $69 $30 $134 $65 $42 $133 $8 $44 $44 $10 $37 $9 $427 $81 $5 $297 $1 $269 $11 $34 $121 $107 $7 $41 $93 $58 $26 $75 $72 $153 $26 $47 $16 $55 $31 $134 $143 $163 $23 $89 $32 $23 $35 $63 $71 $41 $113 $233 $22 $69 $6 $36 $14 $32 $117 $212 $12 $106 $6 $78 $122 $30 $40 $149 $178 $46 $82 $42 $27 $243 $42 $3 $74 $24 $25 $168 $73 $32 $37 $127 $107 $36 $63 $28 $104 $82 $27 $86 $74 $131 $10 $145 $228 $91 $58 $319 $66 $15 $24 $212 $18 $58 $34 $128 $176 $9 $72 $3 $60 $17 $1 $30 $474 $20 $51 $1 $29 $47 $17 $25 $223 $53 $22 $4 $29 $17 $34 $115 $340 $20 $77 $13 $49 $113 $35 $77 $22 $16 $90 $81 $83 $56 $7 $30 $65 $41 $1 $17 

# And now - to the frontier!

Let's see how Frontier models do out of the box; no training, just inference based on their world knowledge.

Tomorrow we will do some training.

In [19]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

In [20]:
print(test[0].summary)

Title: Excess V2 Distortion/Modulation Pedal  
Category: Music Pedals  
Brand: Old Blood Noise  
Description: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  
Details: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.


In [22]:
messages_for(test[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.'}]

In [23]:
# The function for gpt-4.1-nano

def gpt_4__1_nano(item):
    response = completion(model="openai/gpt-4.1-nano", messages=messages_for(item))
    return response.choices[0].message.content

In [24]:
gpt_4__1_nano(test[0])

'$200'

In [25]:
test[0].price

219.0

In [26]:
evaluate(gpt_4__1_nano, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$39 $34 $10 $20 $120 $80 $6 $65 $11 $870 $363 $71 $15 $4 $29 $8 $41 $5 $40 $31 $54 $26 $65 $25 $212 $254 $405 $5 $251 $60 $30 $15 $10 $60 $35 $119 $60 $21 $34 $8 $175 $55 $20 $105 $70 $20 $42 $13 $65 $52 $20 $105 $25 $10 $147 $16 $8 $60 $52 $13 $86 $28 $46 $40 $279 $60 $90 $295 $25 $74 $16 $3 $70 $6 $10 $21 $26 $0 $8 $3 $30 $3 $5 $74 $11 $0 $68 $44 $30 $29 $3 $35 $5 $20 $2 $78 $6 $93 $20 $375 $50 $33 $12 $111 $49 $32 $0 $350 $14 $49 $20 $136 $89 $58 $54 $180 $0 $5 $4 $47 $19 $311 $60 $84 $50 $10 $5 $101 $29 $89 $79 $13 $65 $5 $85 $0 $55 $10 $93 $62 $16 $149 $5 $10 $114 $118 $15 $340 $55 $13 $3 $144 $22 $10 $0 $129 $101 $41 $80 $5 $161 $17 $7 $3 $240 $7 $752 $25 $0 $5 $5 $3 $80 $8 $52 $51 $3 $57 $14 $23 $345 $15 $150 $1 $30 $13 $73 $18 $10 $7 $15 $99 $5 $11 $50 $70 $10 $70 $21 $10 

In [27]:
def claude_opus_4_5(item):
    response = completion(model="anthropic/claude-opus-4-5", messages=messages_for(item))
    return response.choices[0].message.content

In [28]:
evaluate(claude_opus_4_5, test)

  0%|          | 0/200 [00:00<?, ?it/s]

AuthenticationError: litellm.AuthenticationError: Missing Anthropic API Key - A call is being made to anthropic but no key is set either in the environment variables or via params. Please set `ANTHROPIC_API_KEY` in your environment vars

In [29]:
def gemini_3_pro_preview(item):
    response = completion(model="gemini/gemini-3-pro-preview", messages=messages_for(item), reasoning_effort='low')
    return response.choices[0].message.content

In [30]:
evaluate(gemini_3_pro_preview, test, size=50, workers=2)

  0%|          | 0/50 [00:00<?, ?it/s]


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



AuthenticationError: litellm.AuthenticationError: GeminiException - {
  "error": {
    "code": 400,
    "message": "API key not valid. Please pass a valid API key.",
    "status": "INVALID_ARGUMENT",
    "details": [
      {
        "@type": "type.googleapis.com/google.rpc.ErrorInfo",
        "reason": "API_KEY_INVALID",
        "domain": "googleapis.com",
        "metadata": {
          "service": "generativelanguage.googleapis.com"
        }
      },
      {
        "@type": "type.googleapis.com/google.rpc.LocalizedMessage",
        "locale": "en-US",
        "message": "API key not valid. Please pass a valid API key."
      }
    ]
  }
}


In [ ]:
def gemini_2__5_flash_lite(item):
    response = completion(model="gemini/gemini-3.1-flash-lite", messages=messages_for(item))
    return response.choices[0].message.content

In [ ]:
evaluate(gemini_2__5_flash_lite, test)

In [ ]:

def grok_4__1_fast(item):
    response = completion(model="xai/grok-4-1-fast-non-reasoning", messages=messages_for(item), seed=42)
    return response.choices[0].message.content

In [ ]:
evaluate(grok_4__1_fast, test)

In [31]:
# The function for gpt-5.1

def gpt_5__1(item):
    response = completion(model="gpt-5.1", messages=messages_for(item), reasoning_effort='high', seed=42)
    return response.choices[0].message.content


In [32]:
evaluate(gpt_5__1, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$20 $54 $5 $10 $10 $130 $54 $120 $11 $20 $137 $21 $5 $9 $39 $3 $11 $22 $10 $39 $46 $4 $5 $5 $72 $203 $145 $2 $61 $64 $15 $25 $0 $55 $5 $89 $50 $37 $34 $18 $160 $30 $17 $95 $50 $0 $4 $1 $65 $98 $26 $110 $195 $0 $27 $44 $3 $79 $53 $0 $106 $48 $32 $65 $130 $9 $50 $305 $35 $14 $17 $1 $90 $1 $20 $21 $4 $2 $4 $6 $0 $4 $7 $74 $12 $25 $68 $16 $10 $16 $3 $15 $5 $0 $2 $108 $1 $82 $50 $245 $10 $13 $2 $11 $29 $142 $15 $340 $3 $129 $20 $116 $9 $58 $54 $0 $8 $2 $24 $297 $6 $111 $0 $46 $10 $10 $0 $31 $11 $69 $139 $9 $0 $0 $45 $2 $24 $10 $47 $2 $4 $199 $30 $8 $4 $8 $10 $130 $15 $8 $6 $94 $24 $60 $3 $51 $21 $37 $30 $5 $90 $18 $7 $1 $159 $2 $651 $25 $0 $2 $5 $4 $170 $10 $62 $21 $5 $27 $4 $4 $124 $10 $250 $59 $20 $8 $63 $17 $20 $7 $5 $9 $10 $31 $60 $60 $9 $20 $21 $9 